In [0]:
# Lesson: Silver -> Gold
# Step 1: Configuration — create the Gold schema (idempotent) and define layer names.

spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.gold")

silver = "adventure_works.silver"
gold = "adventure_works.gold"
print(f"Silver layer: {silver}")
print(f"Gold layer:   {gold}")

In [0]:
# Step 2: Dimension dim_product — typed copy of silver.product.
# Cast IDs to INT, costs/prices to DECIMAL, dates to TIMESTAMP; drop the technical rowguid column.

from pyspark.sql import functions as F

dim_product = (
    spark.table(f"{silver}.product")
    .select(
        F.col("ProductID").cast("int"),
        F.col("Name"),
        F.col("ProductNumber"),
        F.col("MakeFlag").cast("int"),
        F.col("FinishedGoodsFlag").cast("int"),
        F.col("Color"),
        F.col("SafetyStockLevel").cast("int"),
        F.col("ReorderPoint").cast("int"),
        F.col("StandardCost").cast("decimal(19,4)"),
        F.col("ListPrice").cast("decimal(19,4)"),
        F.col("Size"),
        F.col("SizeUnitMeasureCode"),
        F.col("WeightUnitMeasureCode"),
        F.col("Weight").cast("decimal(8,2)"),
        F.col("DaysToManufacture").cast("int"),
        F.col("ProductLine"),
        F.col("Class"),
        F.col("Style"),
        F.col("ProductSubcategoryID").cast("int"),
        F.col("ProductModelID").cast("int"),
        F.col("SellStartDate").cast("timestamp"),
        F.col("SellEndDate").cast("timestamp"),
        F.col("DiscontinuedDate").cast("timestamp"),
        F.col("ModifiedDate").cast("timestamp"),
    )
)

dim_product.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold}.dim_product")
print(f"Wrote {gold}.dim_product")

In [0]:
# Step 3: Dimension dim_vendor — typed copy of silver.vendor.

from pyspark.sql import functions as F

dim_vendor = (
    spark.table(f"{silver}.vendor")
    .select(
        F.col("BusinessEntityID").cast("int").alias("VendorID"),
        F.col("AccountNumber"),
        F.col("Name").alias("VendorName"),
        F.col("CreditRating").cast("int"),
        F.col("PreferredVendorStatus").cast("int"),
        F.col("ActiveFlag").cast("int"),
        F.col("PurchasingWebServiceURL"),
        F.col("ModifiedDate").cast("timestamp"),
    )
)

dim_vendor.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold}.dim_vendor")
print(f"Wrote {gold}.dim_vendor")

In [0]:
# Step 4: Dimension dim_sales_territory — typed copy of silver.sales_territory.

from pyspark.sql import functions as F

dim_sales_territory = (
    spark.table(f"{silver}.sales_territory")
    .select(
        F.col("TerritoryID").cast("int"),
        F.col("Name").alias("TerritoryName"),
        F.col("CountryRegionCode"),
        F.col("Group").alias("TerritoryGroup"),
        F.col("SalesYTD").cast("decimal(19,4)"),
        F.col("SalesLastYear").cast("decimal(19,4)"),
        F.col("CostYTD").cast("decimal(19,4)"),
        F.col("CostLastYear").cast("decimal(19,4)"),
        F.col("ModifiedDate").cast("timestamp"),
    )
)

dim_sales_territory.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold}.dim_sales_territory")
print(f"Wrote {gold}.dim_sales_territory")

In [0]:
# Step 5: Fact table fact_sales — one row per sales order line (silver.sales_order_detail).
# No sales_order_header source exists, so the fact stays at the line grain.

from pyspark.sql import functions as F

fact_sales = (
    spark.table(f"{silver}.sales_order_detail")
    .select(
        F.col("SalesOrderID").cast("int"),
        F.col("SalesOrderDetailID").cast("int"),
        F.col("CarrierTrackingNumber"),
        F.col("OrderQty").cast("int"),
        F.col("ProductID").cast("int"),
        F.col("SpecialOfferID").cast("int"),
        F.col("UnitPrice").cast("decimal(19,4)"),
        F.col("UnitPriceDiscount").cast("decimal(19,4)"),
        F.col("LineTotal").cast("decimal(19,4)"),
        F.col("ModifiedDate").cast("timestamp").alias("SalesDate"),
    )
)

fact_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold}.fact_sales")
print(f"Wrote {gold}.fact_sales")

In [0]:
# Step 6: Fact table fact_purchases — purchase order lines enriched with header attributes.
# Join silver.purchase_order_detail to silver.purchase_order_header on PurchaseOrderID (LEFT join,
# header is the one side of the key, so the line grain of 8,845 rows is preserved).

from pyspark.sql import functions as F

fact_purchases = (
    spark.table(f"{silver}.purchase_order_detail").alias("d")
    .join(
        spark.table(f"{silver}.purchase_order_header").alias("h"),
        on="PurchaseOrderID",
        how="left",
    )
    .select(
        F.col("PurchaseOrderID").cast("int"),
        F.col("PurchaseOrderDetailID").cast("int"),
        F.col("h.VendorID").cast("int"),
        F.col("d.ProductID").cast("int"),
        F.col("h.OrderDate").cast("timestamp"),
        F.col("h.ShipDate").cast("timestamp"),
        F.col("d.DueDate").cast("timestamp"),
        F.col("d.OrderQty").cast("int"),
        F.col("d.UnitPrice").cast("decimal(19,4)"),
        F.col("d.LineTotal").cast("decimal(19,4)"),
        F.col("d.ReceivedQty").cast("decimal(19,4)"),
        F.col("d.RejectedQty").cast("decimal(19,4)"),
        F.col("d.StockedQty").cast("decimal(19,4)"),
        F.col("d.ModifiedDate").cast("timestamp"),
    )
)

fact_purchases.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold}.fact_purchases")
print(f"Wrote {gold}.fact_purchases")

In [0]:
# Step 7: Aggregate KPI table sales_kpis_by_category_quarter.
# Sales lines joined to products on ProductID; ProductLine mapped to a readable category;
# quarter derived from ModifiedDate as 'yyyy-Qn'; KPIs aggregated per (category, quarter).

from pyspark.sql import functions as F

sales_kpis = (
    spark.table(f"{silver}.sales_order_detail").alias("sod")
    .join(
        spark.table(f"{silver}.product").alias("p"),
        on="ProductID",
        how="left",
    )
    .withColumn(
        "category",
        F.when(F.trim(F.col("p.ProductLine")) == "R", "Road")
        .when(F.trim(F.col("p.ProductLine")) == "M", "Mountain")
        .when(F.trim(F.col("p.ProductLine")) == "S", "Standard")
        .when(F.trim(F.col("p.ProductLine")) == "T", "Touring")
        .otherwise("Other"),  # NULL or empty ProductLine
    )
    .withColumn(
        "quarter",
        F.concat(
            F.date_format(F.col("sod.ModifiedDate").cast("timestamp"), "yyyy"),
            F.lit("-Q"),
            F.date_format(F.col("sod.ModifiedDate").cast("timestamp"), "Q"),
        ),
    )
    .groupBy("category", "quarter")
    .agg(
        F.sum(F.col("sod.LineTotal").cast("decimal(19,4)")).alias("total_revenue"),
        F.sum(F.col("sod.OrderQty").cast("int")).alias("total_order_qty"),
        F.countDistinct(F.col("sod.SalesOrderID")).alias("order_count"),
    )
    .withColumn("avg_order_value", F.col("total_revenue") / F.col("order_count"))
)

sales_kpis.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold}.sales_kpis_by_category_quarter")
print(f"Wrote {gold}.sales_kpis_by_category_quarter")

In [0]:
# Step 8: Verification — row counts for all 6 gold tables, plus the top 10 rows of the KPI table.

gold_tables = [
    "dim_product",
    "dim_vendor",
    "dim_sales_territory",
    "fact_sales",
    "fact_purchases",
    "sales_kpis_by_category_quarter",
]

row_counts = spark.createDataFrame(
    [(t, spark.table(f"{gold}.{t}").count()) for t in gold_tables],
    ["gold_table", "row_count"],
)
display(row_counts)

display(
    spark.table(f"{gold}.sales_kpis_by_category_quarter")
    .orderBy("quarter", "category")
    .limit(10)
)